**INPUT OF THE DATASET OF GPT, CLAUDE, GEMINI AND BAWE CORPUS**

Links of the datasets:

- ChatGPT (MGTBench): https://github.com/Y-L-LIU/MGTBench-2.0

- Claude Dataset: https://huggingface.co/datasets/QuietImpostor/Claude-3-Opus-Claude-3.5-Sonnnet-9k

- Gemini Dataset: https://www.kaggle.com/datasets/etiennekaiser/gemini-pro-llm-daigt-dataset

- BAWE Corpus: https://ota.bodleian.ox.ac.uk/repository/xmlui/handle/20.500.12024/2539

Raw outputs (used by `data_cleaning.ipynb`):
- `data/raw/ai/gemini_essays_v1.csv`
- `data/raw/ai/claude_dataset.csv`
- `data/raw/ai/mgtbench_ai_dataset.csv`
- `data/raw/human/bawe_dataset.csv`

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_colwidth', 150)

# Match path layout used in data_cleaning.ipynb
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR.parent.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_AI_DIR = RAW_DIR / "ai"
RAW_HUMAN_DIR = RAW_DIR / "human"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_AI_DIR = PROCESSED_DIR / "ai"
PROCESSED_HUMAN_DIR = PROCESSED_DIR / "human"

for path in (RAW_AI_DIR, RAW_HUMAN_DIR, PROCESSED_AI_DIR, PROCESSED_HUMAN_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Data directories ready:")
print(f"  Raw AI:          {RAW_AI_DIR.resolve()}")
print(f"  Raw Human:       {RAW_HUMAN_DIR.resolve()}")
print(f"  Processed AI:    {PROCESSED_AI_DIR.resolve()}")
print(f"  Processed Human: {PROCESSED_HUMAN_DIR.resolve()}")

In [ ]:
import kaggle
from dotenv import load_dotenv

print("1/4 Loading Gemini-Pro Dataset...")

load_dotenv(PROJECT_ROOT / ".env.local")
load_dotenv(PROJECT_ROOT / ".env")

gemini_target_file = RAW_AI_DIR / "gemini_essays_v1.csv"

if not gemini_target_file.exists():
    print("Downloading Gemini-Pro dataset from Kaggle...")
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        "etiennekaiser/gemini-pro-llm-daigt-dataset",
        path=str(RAW_AI_DIR),
        unzip=True,
    )
    for csv_file in RAW_AI_DIR.glob("*.csv"):
        if csv_file.name != "gemini_essays_v1.csv":
            csv_file.rename(gemini_target_file)
            print(f"Renamed {csv_file.name} -> gemini_essays_v1.csv")
            break
else:
    print(f"Using existing file: {gemini_target_file.resolve()}")

if gemini_target_file.exists():
    df_gemini = pd.read_csv(gemini_target_file)
    print(f"Gemini dataset shape: {df_gemini.shape}")
    print(f"Columns: {df_gemini.columns.tolist()}")
    display(df_gemini.head(10))
else:
    print(f"Error: could not find gemini_essays_v1.csv in {RAW_AI_DIR}")

In [ ]:
print("2/4 Loading Claude-3 Dataset from Hugging Face...")

claude_file_path = RAW_AI_DIR / "claude_dataset.csv"

if not claude_file_path.exists():
    df_claude = pd.read_json(
        "hf://datasets/QuietImpostor/Claude-3-Opus-Claude-3.5-Sonnnet-9k/"
        "Claude-3-Opus-and-Claude-3-5-Sonnet-9k-ShareGPT.jsonl",
        lines=True,
    )
    print(f"Dataset loaded from Hugging Face. Shape: {df_claude.shape}")
    df_claude.to_csv(claude_file_path, index=False, encoding="utf-8")
    print(f"Saved Claude dataset to: {claude_file_path.resolve()}")
else:
    df_claude = pd.read_csv(claude_file_path)
    print(f"Using existing file: {claude_file_path.resolve()}")
    print(f"Shape: {df_claude.shape}")

display(df_claude.head(10))

In [ ]:
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

fs = HfFileSystem()

CATEGORIES = [
    "Physics", "Medicine", "Biology", "Electrical_engineering", "Computer_science",
    "Literature", "History", "Education", "Art", "Law", "Management",
    "Philosophy", "Economy", "Math", "Statistics", "Chemistry",
]

# GPT-3.5 only — matches scope in data_cleaning.ipynb
AI_MODELS = ["gpt35_new"]
mgtbench_ai_path = RAW_AI_DIR / "mgtbench_ai_dataset.csv"

print("3/4 Loading MGTBench ChatGPT (GPT-3.5) dataset...")

if not mgtbench_ai_path.exists():
    all_ai_data = []
    print("Downloading MGTBench AI datasets from Hugging Face...")

    for model in tqdm(AI_MODELS, desc="AI Models"):
        for cat in CATEGORIES:
            file_path = f"datasets/AITextDetect/AI_Polish_clean/{model}/{cat}_task3.json"
            if fs.exists(file_path):
                try:
                    df = pd.read_json(f"hf://{file_path}")
                    all_ai_data.append(df)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

    if not all_ai_data:
        raise RuntimeError("No MGTBench AI files were loaded from Hugging Face.")

    df_mgtbench_ai = pd.concat(all_ai_data, ignore_index=True)
    df_mgtbench_ai.to_csv(mgtbench_ai_path, index=False, encoding="utf-8")
    print(f"Saved MGTBench AI dataset to: {mgtbench_ai_path.resolve()}")
else:
    df_mgtbench_ai = pd.read_csv(mgtbench_ai_path)
    print(f"Using existing file: {mgtbench_ai_path.resolve()}")

print(f"MGTBench AI shape: {df_mgtbench_ai.shape}")
print(f"Columns: {df_mgtbench_ai.columns.tolist()}")
display(df_mgtbench_ai.head(10))

In [ ]:
import requests
import zipfile

bawe_zip_path = RAW_HUMAN_DIR / "bawe.zip"
bawe_extract_dir = RAW_HUMAN_DIR / "bawe"
bawe_corpus_url = (
    "https://ota.bodleian.ox.ac.uk/repository/xmlui/bitstream/"
    "20.500.12024/2539/3/2539.zip"
)

print("4/4 Loading BAWE Corpus Dataset...")

if bawe_zip_path.exists() and not zipfile.is_zipfile(bawe_zip_path):
    print("Removing corrupted zip file...")
    bawe_zip_path.unlink()

if not bawe_zip_path.exists():
    print("Downloading BAWE corpus zip...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        response = requests.get(
            bawe_corpus_url, headers=headers, stream=True, timeout=120
        )
        response.raise_for_status()
        with open(bawe_zip_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded BAWE zip to: {bawe_zip_path.resolve()}")
    except Exception as e:
        print(f"Download failed: {e}")
        if bawe_zip_path.exists():
            bawe_zip_path.unlink()
else:
    print(f"Using existing zip: {bawe_zip_path.resolve()}")

if bawe_zip_path.exists() and zipfile.is_zipfile(bawe_zip_path):
    if not bawe_extract_dir.exists() or not any(bawe_extract_dir.iterdir()):
        print("Extracting BAWE corpus zip...")
        with zipfile.ZipFile(bawe_zip_path, "r") as zip_ref:
            zip_ref.extractall(bawe_extract_dir)
        print(f"Extracted to: {bawe_extract_dir.resolve()}")
    else:
        print(f"Extracted corpus ready at: {bawe_extract_dir.resolve()}")

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from utils.cleaning import ingest_bawe_dataset

bawe_corpus_dir = bawe_extract_dir / "download" / "CORPUS_ByDisc"
bawe_csv_path = RAW_HUMAN_DIR / "bawe_dataset.csv"

if not bawe_corpus_dir.exists():
    print(f"BAWE corpus XML not found at: {bawe_corpus_dir.resolve()}")
    print("Run the previous cell to download and extract the zip first.")
elif not bawe_csv_path.exists():
    df_bawe = ingest_bawe_dataset(bawe_corpus_dir, bawe_csv_path)
    if df_bawe is not None:
        print(f"BAWE dataset shape: {df_bawe.shape}")
        display(df_bawe.head(10))
else:
    df_bawe = pd.read_csv(bawe_csv_path)
    print(f"Using existing file: {bawe_csv_path.resolve()}")
    print(f"BAWE dataset shape: {df_bawe.shape}")
    display(df_bawe.head(10))